In [31]:
import pandas as pd

# Load dataset
df = pd.read_csv('../data/interim/covid19_tweets.csv')

In [32]:
def detect_variable_types(df, target_col=None):
    """Automatically detect variable types in a DataFrame."""
    types = {}
    
    for col in df.columns:
        if col == target_col:
            types[col] = 'target'
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            types[col] = 'datetime'
        elif pd.api.types.is_numeric_dtype(df[col]):
            if df[col].nunique() == 2:
                types[col] = 'binary'
            else:
                types[col] = 'numerical'
        elif df[col].nunique() < 20:
            # Threshold for categorical
            types[col] = 'categorical'
        elif df[col].dtype == 'object':
            # Check if it's text (long strings) or categorical
            avg_length = df[col].astype(str).str.len().mean()
            if avg_length > 50:
                types[col] = 'text'
            else:
                types[col] = 'categorical'
        else:
            types[col] = 'other'
    
    return types

In [33]:
var_types = detect_variable_types(df, target_col=None)
var_types

{'user_name': 'categorical',
 'user_location': 'categorical',
 'user_description': 'text',
 'user_created': 'categorical',
 'user_followers': 'numerical',
 'user_friends': 'numerical',
 'user_favourites': 'numerical',
 'user_verified': 'binary',
 'date': 'categorical',
 'text': 'text',
 'hashtags': 'categorical',
 'source': 'categorical',
 'is_retweet': 'numerical'}

# Data Preparation

In [ ]:
# Before proceeding, we need to handle user_name because it is a sensitive feature.

# Mapping from user_name to an integer ID
unique_users = df['user_name'].astype(str).unique()
user_id_map = {name: i for i, name in enumerate(unique_users)}

df['user_id'] = df['user_name'].astype(str).map(user_id_map)

df[['user_name', 'user_id']].head(10)

,user_name,user_id
0,ᏉᎥ☻լꂅϮ,0
1,Tom Basile 🇺🇸,1
2,Time4fisticuffs,2
3,ethel mertz,3
4,DIPR-J&K,4
5,🎹 Franz Schubert,5
6,hr bartender,6
7,Derbyshire LPC,7
8,Prathamesh Bendre,8
9,Member of Christ 🇨🇳🇺🇸🇮🇳🇮🇩🇧🇷🇳🇬🇧🇩🇷🇺,9


To align with AI‑ethics and data‑minimisation principles, this project takes extra steps to protect user privacy and avoid unnecessary processing of potentially sensitive information.

First, the `user_name` column is not used directly in modelling. Where user‑level information is needed (for example, to count tweets per account), usernames are mapped to integer IDs (`user_id`) and the original user_name values are then dropped from the working dataset. This pseudonymisation preserves the ability to reason about user‑level patterns without retaining direct identifiers.

Second, the `user_description` column, which contains free‑text profile bios, is removed entirely and not used as a feature. These descriptions often reveal roles (e.g., “columnist”, “professor”), affiliations (“official account of…”, “news alerts from…”), and sometimes religious, political, or other sensitive attributes. While such information could potentially correlate with tweet tone, it is not strictly necessary for the core task of classifying each tweet as fear‑mongering, calm‑negative/concerned, or neutral/factual. To treat user data with appropriate sensitivity and protection, the project restricts itself to tweet text and non‑identifying metadata (such as timestamps and simple engagement counts), documenting this choice as an explicit application of data‑minimisation and privacy‑by‑design principles.

Thirdly, the `source` column will be dropped because:
- It primarily encodes user device's information.
- It has no clear, theory‑backed link to tweet tone or factuality.

In [37]:
# Dropping user_name column because we no longer need it.
df = df.drop(columns=['user_name'])

In [39]:
df['user_description'].head(24)

0     wednesday addams as a disney princess keepin i...
1     Husband, Father, Columnist & Commentator. Auth...
2     #Christian #Catholic #Conservative #Reagan #Re...
3     #Browns #Indians #ClevelandProud #[]_[] #Cavs ...
4     🖊️Official Twitter handle of Department of Inf...
5     🎼  #Новоро́ссия #Novorossiya #оставайсядома #S...
6     Workplace tips and advice served up in a frien...
7                                                   NaN
8      A poet, reiki practitioner and a student of law.
9     Just as the body is one & has many members, & ...
10                                                  NaN
11    I'm Motalib Mia, Logo -Logo Designer - Brandin...
12    My ink "My Way...No Regrets"\nAlways Make Happ...
13    Official account of the Africa Youth Advisory ...
14                     Breaking news alerts from India.
15    strive to promote Truth with Integrity.\nhttps...
16    Individual tweeting about significant happenin...
17    Progressive mind. Flemish. Into movies, po

In [ ]:
# Drop profile descriptions to prevent use of personal/sensitive data
df = df.drop(columns=['user_description'])

In [49]:
df.source.head(22)

0      Twitter for iPhone
1     Twitter for Android
2     Twitter for Android
3      Twitter for iPhone
4     Twitter for Android
5         Twitter Web App
6                  Buffer
7               TweetDeck
8     Twitter for Android
9      Twitter for iPhone
10        Twitter Web App
11        Twitter Web App
12        Twitter Web App
13        Twitter Web App
14        Twitter Web App
15    Twitter for Android
16     Twitter for iPhone
17    Twitter for Android
18        Twitter Web App
19       Twitter for iPad
20     Twitter for iPhone
21        Africa Newsroom
Name: source, dtype: object

In [51]:
df.source.head(22)

0      Twitter for iPhone
1     Twitter for Android
2     Twitter for Android
3      Twitter for iPhone
4     Twitter for Android
5         Twitter Web App
6                  Buffer
7               TweetDeck
8     Twitter for Android
9      Twitter for iPhone
10        Twitter Web App
11        Twitter Web App
12        Twitter Web App
13        Twitter Web App
14        Twitter Web App
15    Twitter for Android
16     Twitter for iPhone
17    Twitter for Android
18        Twitter Web App
19       Twitter for iPad
20     Twitter for iPhone
21        Africa Newsroom
Name: source, dtype: object

In [52]:
# Dropping source column due to irrelevancy to future training of the model
df = df.drop(columns=['source'])

In [53]:
# Names of columns in our dataset
df.columns

Index(['user_location', 'user_created', 'user_followers', 'user_friends',
       'user_favourites', 'user_verified', 'date', 'text', 'hashtags',
       'is_retweet', 'user_id'],
      dtype='object')

In [40]:
# Basic missing value overview
def quick_missing_analysis(df):
    missing_counts = df.isnull().sum()
    missing_percent = (missing_counts / len(df)) * 100
    
    print("Missing Value Summary:")
    for col in df.columns:
        if missing_counts[col] > 0:
            print(f"  {col}: {missing_counts[col]} missing ({missing_percent[col]:.1f}%)")

quick_missing_analysis(df)

Missing Value Summary:
  user_location: 36771 missing (20.5%)
  user_description: 10286 missing (5.7%)
  hashtags: 51334 missing (28.7%)
  source: 77 missing (0.0%)


In [41]:
def basic_range_check_tweets(df):
    """Quick sanity checks for numerical / binary columns in the COVID19 tweets dataset."""
    numeric_cols = df.select_dtypes(include=['number']).columns
    
    for col in numeric_cols:
        col_min = df[col].min()
        col_max = df[col].max()
        print(f"{col}: min={col_min}, max={col_max}")
        
        # Count / size fields should not be negative
        if col in ['user_followers', 'user_friends', 'user_favourites']:
            if col_min < 0:
                print(f"  ⚠️ Suspicious negative count values detected in {col}")
            # Optional: flag extremely large values as potential outliers
            q99 = df[col].quantile(0.99)
            if q99 > 1e6:
                print(f"  ⚠️ Very large values in {col} (99th percentile = {q99:.0f}) – check for outliers/bots")
        
        # Binary indicators should only take 0/1 (or 0/1/NaN)
        if col in ['user_verified', 'is_retweet']:
            unique_vals = sorted(df[col].dropna().unique())
            print(f"  unique values in {col}: {unique_vals}")
            if not set(unique_vals).issubset({0, 1}):
                print(f"  ⚠️ {col} contains non-binary values – needs cleaning or recoding")
                
basic_range_check_tweets(df)

user_followers: min=0, max=49442559
  ⚠️ Very large values in user_followers (99th percentile = 3293396) – check for outliers/bots
user_friends: min=0, max=497363
user_favourites: min=0, max=2047197
user_id: min=0, max=92275
